# Self-Speculative Decoding for Speech LLMs

This notebook demonstrates **self-speculative decoding** for [granite-4.0-1b-speech](https://huggingface.co/ibm-granite/granite-4.0-1b-speech) cf. this [paper](https://arxiv.org/abs/2603.11243)
1. A fast **CTC encoder** generates draft transcriptions
2. The **LLM verifies** the draft in a single forward pass
3. If verification fails, we **fall back** to autoregressive decoding

This approach can significantly speed up inference by avoiding full AR decoding for "easy" utterances.

## Setup

In [16]:
import math
import torch
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, models
from huggingface_hub import hf_hub_download
import soundfile as sf
import librosa
import IPython.display as ipd

# Ensure granite_speech is available
assert hasattr(models, "granite_speech"), "Please install transformers with granite_speech support"

torch.set_float32_matmul_precision('high')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## Load Model and Processor

In [17]:
MODEL_ID = "ibm-granite/granite-4.0-1b-speech"

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

# Use bfloat16 on CUDA, float32 on CPU
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, 
    torch_dtype=dtype
).to(device)
model.eval()

# Get logits scaling factor if present
logits_scaling = getattr(model.language_model.config, 'logits_scaling', 1.0)
print(f"Model loaded on {device} with dtype {dtype}. Logits scaling: {logits_scaling}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on cpu with dtype torch.float32. Logits scaling: 8


## Setup Chat Template and Cache Prompt Embeddings

We pre-compute embeddings for the static parts of the prompt (prefix and suffix) to avoid redundant computation.

In [18]:
# Define the instruction
text_instruction = "<|audio|>can you transcribe the speech into a written format?"

# Build chat message and apply template
message = [{"role": "user", "content": text_instruction}]
text_prompt = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

print("Full prompt template:")
print(text_prompt)
print()

Full prompt template:
USER: <|audio|>can you transcribe the speech into a written format?
 ASSISTANT:



In [19]:
# Split prompt into prefix (before audio) and suffix (after audio)
prompt_prefix, prompt_suffix = text_prompt.split("<|audio|>")

print(f"Prefix: {repr(prompt_prefix)}")
print(f"Suffix: {repr(prompt_suffix)}")

Prefix: 'USER: '
Suffix: 'can you transcribe the speech into a written format?\n ASSISTANT:'


In [20]:
# Cache prompt embeddings for reuse
embed_layer = model.language_model.get_input_embeddings()

prefix_ids = tokenizer.encode(prompt_prefix, add_special_tokens=False)
suffix_ids = tokenizer.encode(prompt_suffix, add_special_tokens=False)

cached_prefix_embeds = embed_layer(torch.tensor([prefix_ids], device=device))
cached_suffix_embeds = embed_layer(torch.tensor([suffix_ids], device=device))

print(f"Prefix tokens: {len(prefix_ids)}, Suffix tokens: {len(suffix_ids)}")
print(f"Cached prefix embeds shape: {cached_prefix_embeds.shape}")
print(f"Cached suffix embeds shape: {cached_suffix_embeds.shape}")

Prefix tokens: 3, Suffix tokens: 15
Cached prefix embeds shape: torch.Size([1, 3, 2048])
Cached suffix embeds shape: torch.Size([1, 15, 2048])


## Configuration

In [21]:
# Speculative decoding thresholds
CONFIDENCE_THRESHOLD = 0.1  # LLM verification acceptance threshold
CTC_THRESHOLD = 0.7          # CTC entropy threshold for direct acceptance

# Generation parameters
MAX_NEW_TOKENS = 200
NUM_BEAMS = 1

# Audio parameters
HOP_LENGTH = 160

## Load Sample Audio

In [22]:
# Load sample audio from the granite-4.0-1b-speech model card
from huggingface_hub import hf_hub_download
import soundfile as sf

audio_path = hf_hub_download(repo_id=MODEL_ID, filename="multilingual_sample.wav")
audio, sample_rate = sf.read(audio_path)

# Resample to 16kHz if needed
if sample_rate != 16000:
    audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=16000)

print(f"Audio file: {audio_path}")
print(f"Audio length: {len(audio) / 16000:.2f}s")
ipd.Audio(audio, rate=16000)

Audio file: /Users/saon/.cache/huggingface/hub/models--ibm-granite--granite-4.0-1b-speech/snapshots/3d1f193957c13244f2ceeb0dbb7f9f3dea4c8032/multilingual_sample.wav
Audio length: 24.94s


## Step 1: CTC Decoding with Entropy-Based Confidence

The CTC decoder provides a fast initial transcription along with confidence scores based on entropy.

In [23]:
@torch.no_grad()
def ctc_decode(audios):
    """CTC decode with entropy-based confidence.
    
    Returns:
        ctc_texts: List of decoded texts
        ctc_entropies: List of max entropy values (confidence measure)
        embeddings: Encoder embeddings for later use
        embed_lengths: Length of each embedding sequence
    """
    texts = [text_prompt] * len(audios)
    model_inputs = processor(texts, audios, device=device, return_tensors="pt").to(device)
    
    # Use autocast only on CUDA
    if device == "cuda":
        with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
            encoder_output = model.encoder(model_inputs["input_features"])
            embeddings = encoder_output.last_hidden_state if hasattr(encoder_output, 'last_hidden_state') else encoder_output
            ctc_logits = model.encoder.out(embeddings)
    else:
        encoder_output = model.encoder(model_inputs["input_features"].to(dtype))
        embeddings = encoder_output.last_hidden_state if hasattr(encoder_output, 'last_hidden_state') else encoder_output
        ctc_logits = model.encoder.out(embeddings)
    
    ctc_probs = F.softmax(ctc_logits.float(), dim=-1)
    _, idx_batch = ctc_probs.max(dim=-1)
    
    # Compute entropy as confidence measure
    entropy = -(ctc_probs * torch.log(ctc_probs + 1e-10)).sum(dim=-1)
    
    ctc_texts, ctc_entropies, embed_lengths = [], [], []
    for i, idx in enumerate(idx_batch):
        # Remove consecutive duplicates and blanks
        dedup = torch.unique_consecutive(idx, dim=-1)
        non_blank = dedup[dedup > 0].tolist()
        
        # Convert to text (character-level CTC)
        ctc_texts.append(''.join(chr(c) for c in non_blank))
        ctc_entropies.append(entropy[i].max().item() if non_blank else float('inf'))
        embed_lengths.append(len(audios[i]) // HOP_LENGTH // 2 + 1)
    
    return ctc_texts, ctc_entropies, embeddings, embed_lengths

In [24]:
# Run CTC decoding
ctc_texts, ctc_entropies, embeddings, embed_lengths = ctc_decode([audio])

print(f"CTC text: {ctc_texts[0]}")
print(f"CTC entropy (max): {ctc_entropies[0]:.4f}")
print(f"Embedding shape: {embeddings.shape}")
print(f"\nCTC threshold: {CTC_THRESHOLD}")
print(f"Accept directly? {ctc_entropies[0] <= CTC_THRESHOLD}")

CTC text: for timothy was a spoilet cat and he allowed no one to interfere everybody waited upon him moving their chairs even for he was monarch of the hearth dinarzade la nuit suivante appela sa soeur quand il en fut temps si vous ne dormez pas ma soeur lui dit-elle je vous prie en attendant le jour qui paraîtra bientôt de continuer le compte du pêcheur 
CTC entropy (max): 1.0486
Embedding shape: torch.Size([1, 1247, 1024])

CTC threshold: 0.7
Accept directly? False


## Step 2: LLM Verification

If the CTC confidence is not high enough, we verify the draft with the LLM in a single forward pass.

The key insight is that we can compute the probability of each CTC token given the context by running the LLM once with the full sequence.

In [25]:
@torch.no_grad()
def verify(ctc_texts, embeddings, embed_lengths):
    """Verify CTC outputs with LLM in a single forward pass.
    
    Returns:
        results: List of (accepted, text) tuples
        audio_embeds: Projected audio embeddings
        proj_lengths: Length of projected embeddings
    """
    batch_sz = len(ctc_texts)
    
    # Tokenize CTC outputs
    ctc_token_ids = []
    for text in ctc_texts:
        text = text.strip() if text else ""
        ctc_token_ids.append(tokenizer.encode(text, add_special_tokens=False) if text else [])
    
    # Project audio embeddings
    if device == "cuda":
        with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
            audio_embeds = model.projector(embeddings)
    else:
        audio_embeds = model.projector(embeddings)
    max_proj_len = audio_embeds.shape[1]
    
    # Calculate projected lengths
    window_size, downsample_rate = model.config.window_size, model.config.downsample_rate
    num_queries = window_size // downsample_rate
    proj_lengths = [min(math.ceil(enc_len / window_size) * num_queries, max_proj_len) 
                   for enc_len in embed_lengths]
    
    if not any(ctc_token_ids):
        return [(False, ctc_texts[i]) for i in range(batch_sz)], audio_embeds, proj_lengths
    
    audio_token_id = model.config.audio_token_id
    all_input_ids, prompt_lens, audio_ranges = [], [], []
    
    # Build input sequences: [prefix] + [audio_tokens] + [suffix] + [ctc_output]
    for i, proj_len in enumerate(proj_lengths):
        audio_start = len(prefix_ids)
        audio_ranges.append((audio_start, audio_start + proj_len))
        prompt_part = prefix_ids + [audio_token_id] * proj_len + suffix_ids
        prompt_lens.append(len(prompt_part))
        all_input_ids.append(prompt_part + ctc_token_ids[i])
    
    # Pad sequences
    max_len = max(len(ids) for ids in all_input_ids)
    padded_ids = torch.full((batch_sz, max_len), tokenizer.pad_token_id, dtype=torch.long, device=device)
    attn_mask = torch.zeros(batch_sz, max_len, dtype=torch.long, device=device)
    for i, ids in enumerate(all_input_ids):
        padded_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=device)
        attn_mask[i, :len(ids)] = 1
    
    # Get embeddings and replace audio placeholder with actual audio embeddings
    inputs_embeds = model.language_model.get_input_embeddings()(padded_ids)
    for i in range(batch_sz):
        s, e = audio_ranges[i]
        inputs_embeds[i, s:e, :] = audio_embeds[i, :e-s, :]
    
    # Run LLM forward pass
    if device == "cuda":
        with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
            hidden = model.language_model.model(
                attention_mask=attn_mask, 
                inputs_embeds=inputs_embeds, 
                use_cache=False
            ).last_hidden_state
    else:
        hidden = model.language_model.model(
            attention_mask=attn_mask, 
            inputs_embeds=inputs_embeds, 
            use_cache=False
        ).last_hidden_state
    
    # Gather hidden states at verification positions
    sample_idx, pos_idx, ctc_flat = [], [], []
    sample_ranges, sample_valid = [], []
    offset = 0
    
    for i in range(batch_sz):
        ctc_tokens = ctc_token_ids[i]
        if not ctc_tokens or prompt_lens[i] - 1 + len(ctc_tokens) > hidden.shape[1]:
            sample_ranges.append((offset, offset))
            sample_valid.append(False)
            continue
        verify_start = prompt_lens[i] - 1
        for k in range(len(ctc_tokens)):
            sample_idx.append(i)
            pos_idx.append(verify_start + k)
            ctc_flat.append(ctc_tokens[k])
        sample_ranges.append((offset, offset + len(ctc_tokens)))
        sample_valid.append(True)
        offset += len(ctc_tokens)
    
    if pos_idx:
        gathered = hidden[torch.tensor(sample_idx, device=device), 
                         torch.tensor(pos_idx, device=device), :]
        if device == "cuda":
            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits = model.language_model.lm_head(gathered) / logits_scaling
        else:
            logits = model.language_model.lm_head(gathered) / logits_scaling
        probs = F.softmax(logits.float(), dim=-1)
        ctc_probs = probs[torch.arange(len(ctc_flat), device=device), 
                         torch.tensor(ctc_flat, device=device)]
    
    # Check if all tokens pass the confidence threshold
    results = []
    for i in range(batch_sz):
        s, e = sample_ranges[i]
        if not sample_valid[i]:
            results.append((False, ctc_texts[i]))
            continue
        token_probs = ctc_probs[s:e]
        accepted = (token_probs >= CONFIDENCE_THRESHOLD).all().item()
        results.append((accepted, ctc_texts[i]))
    
    return results, audio_embeds, proj_lengths

In [26]:
# Run verification
results, audio_embeds, proj_lengths = verify(ctc_texts, embeddings, embed_lengths)

accepted, text = results[0]
print(f"Verification result: {'ACCEPTED' if accepted else 'REJECTED'}")
print(f"Text: {text}")

Verification result: REJECTED
Text: for timothy was a spoilet cat and he allowed no one to interfere everybody waited upon him moving their chairs even for he was monarch of the hearth dinarzade la nuit suivante appela sa soeur quand il en fut temps si vous ne dormez pas ma soeur lui dit-elle je vous prie en attendant le jour qui paraîtra bientôt de continuer le compte du pêcheur 


## Step 3: Autoregressive Fallback

If verification fails, we fall back to standard autoregressive decoding.

In [27]:
@torch.no_grad()
def fallback(audio_embeds, indices, proj_lengths):
    """AR fallback for samples that failed verification."""
    if not indices:
        return []
    
    batch_sz = len(indices)
    hidden_dim = audio_embeds.shape[-1]
    all_embeds, all_lengths = [], []
    
    for i in indices:
        sample_embeds = audio_embeds[i, :proj_lengths[i], :].unsqueeze(0)
        combined = torch.cat([cached_prefix_embeds, sample_embeds, cached_suffix_embeds], dim=1)
        all_embeds.append(combined.squeeze(0))
        all_lengths.append(combined.shape[1])
    
    # Pad sequences (left-padding for generation)
    max_len = max(all_lengths)
    padded = torch.zeros(batch_sz, max_len, hidden_dim, device=device, dtype=audio_embeds.dtype)
    attn_mask = torch.zeros(batch_sz, max_len, dtype=torch.long, device=device)
    for i, (emb, length) in enumerate(zip(all_embeds, all_lengths)):
        padded[i, max_len - length:] = emb
        attn_mask[i, max_len - length:] = 1
    
    # Generate
    outputs = model.language_model.generate(
        inputs_embeds=padded, 
        attention_mask=attn_mask,
        bos_token_id=tokenizer.bos_token_id, 
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id, 
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS, 
        early_stopping=NUM_BEAMS > 1,
        do_sample=False, 
        use_cache=True
    )
    
    return [tokenizer.decode(outputs[i], skip_special_tokens=True) for i in range(batch_sz)]

In [28]:
# Demo fallback (even if verification passed)
fallback_result = fallback(audio_embeds, [0], proj_lengths)
print(f"Fallback result: {fallback_result[0]}")

Fallback result: for timothy was a spoiled cat and he allowed no one to interfere everybody waited upon him moving their chairs even for he was monarch of the hearth dinarzade la nuit suivante appela sa soeur quand il en fut temps si vous ne dormez pas ma soeur lui dit-elle je vous prie en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur


## Complete Pipeline

Now let's put it all together into a complete speculative decoding pipeline.

In [29]:
import time

@torch.no_grad()
def speculative_transcribe(audios):
    """Complete speculative decoding pipeline.
    
    Args:
        audios: List of audio arrays (16kHz)
        
    Returns:
        predictions: List of transcriptions
        stats: Dictionary with timing and acceptance stats
    """
    batch_sz = len(audios)
    start_time = time.time()
    
    # Step 1: CTC decode
    ctc_start = time.time()
    ctc_texts, ctc_entropies, embeddings, embed_lengths = ctc_decode(audios)
    ctc_time = time.time() - ctc_start
    
    # Step 2: Gate by CTC entropy
    predictions = [None] * batch_sz
    verify_idx = []
    ctc_accepted = 0
    
    for i, (text, ent) in enumerate(zip(ctc_texts, ctc_entropies)):
        if ent <= CTC_THRESHOLD and text.strip():
            predictions[i] = text.strip()
            ctc_accepted += 1
        else:
            verify_idx.append(i)
    
    # Step 3: Verify remaining
    verify_time = 0
    llm_accepted = 0
    
    if verify_idx:
        verify_start = time.time()
        verify_emb = embeddings[verify_idx]
        verify_lens = [embed_lengths[i] for i in verify_idx]
        verify_texts = [ctc_texts[i] for i in verify_idx]
        
        results, audio_embeds, proj_lengths = verify(verify_texts, verify_emb, verify_lens)
        verify_time = time.time() - verify_start
        
        fail_idx = []
        for j, (accepted, text) in enumerate(results):
            i = verify_idx[j]
            if accepted:
                predictions[i] = text.strip()
                llm_accepted += 1
            else:
                fail_idx.append(j)
        
        # Step 4: Fallback
        if fail_idx:
            fallback_start = time.time()
            fallback_texts = fallback(audio_embeds, fail_idx, proj_lengths)
            fallback_time = time.time() - fallback_start
            for k, j in enumerate(fail_idx):
                predictions[verify_idx[j]] = fallback_texts[k]
        else:
            fallback_time = 0
    else:
        fallback_time = 0
    
    total_time = time.time() - start_time
    
    stats = {
        'total_time': total_time,
        'ctc_time': ctc_time,
        'verify_time': verify_time,
        'fallback_time': fallback_time,
        'ctc_accepted': ctc_accepted,
        'llm_accepted': llm_accepted,
        'fallback_count': batch_sz - ctc_accepted - llm_accepted,
        'batch_size': batch_sz
    }
    
    return predictions, stats

In [30]:
# Run the complete pipeline
predictions, stats = speculative_transcribe([audio])

print("=" * 50)
print("SPECULATIVE DECODING RESULTS")
print("=" * 50)
print(f"\nTranscription: {predictions[0]}")
print(f"\nTiming:")
print(f"  CTC decode:    {stats['ctc_time']*1000:.1f}ms")
print(f"  LLM verify:    {stats['verify_time']*1000:.1f}ms")
print(f"  AR fallback:   {stats['fallback_time']*1000:.1f}ms")
print(f"  Total:         {stats['total_time']*1000:.1f}ms")
print(f"\nAcceptance:")
print(f"  CTC accepted:  {stats['ctc_accepted']}/{stats['batch_size']}")
print(f"  LLM accepted:  {stats['llm_accepted']}/{stats['batch_size']}")
print(f"  Fallback:      {stats['fallback_count']}/{stats['batch_size']}")

SPECULATIVE DECODING RESULTS

Transcription: for timothy was a spoiled cat and he allowed no one to interfere everybody waited upon him moving their chairs even for he was monarch of the hearth dinarzade la nuit suivante appela sa soeur quand il en fut temps si vous ne dormez pas ma soeur lui dit-elle je vous prie en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur

Timing:
  CTC decode:    2415.9ms
  LLM verify:    1110.1ms
  AR fallback:   23547.2ms
  Total:         27073.2ms

Acceptance:
  CTC accepted:  0/1
  LLM accepted:  0/1
  Fallback:      1/1


## Comparison: Speculative vs Standard AR Decoding

In [31]:
@torch.no_grad()
def standard_transcribe(audios):
    """Standard autoregressive decoding for comparison."""
    start_time = time.time()
    
    texts = [text_prompt] * len(audios)
    model_inputs = processor(texts, audios, device=device, return_tensors="pt").to(device)
    
    outputs = model.generate(
        **model_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False
    )
    
    predictions = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
    total_time = time.time() - start_time
    
    return predictions, total_time

In [32]:
# Compare methods
print("Running comparison...\n")

# Standard AR
ar_predictions, ar_time = standard_transcribe([audio])

# Speculative
spec_predictions, spec_stats = speculative_transcribe([audio])

print("COMPARISON")
print("=" * 50)
print(f"\nStandard AR:")
print(f"  Time: {ar_time*1000:.1f}ms")
print(f"  Output: {ar_predictions[0]}")
print(f"\nSpeculative:")
print(f"  Time: {spec_stats['total_time']*1000:.1f}ms")
print(f"  Output: {spec_predictions[0]}")
print(f"\nSpeedup: {ar_time / spec_stats['total_time']:.2f}x")

Running comparison...

COMPARISON

Standard AR:
  Time: 25990.7ms
  Output: USER: can you transcribe the speech into a written format?
 ASSISTANT:for timothy was a spoiled cat and he allowed no one to interfere everybody waited upon him moving their chairs even for he was monarch of the hearth dinarzade la nuit suivante appela sa soeur quand il en fut temps si vous ne dormez pas ma soeur lui dit-elle je vous prie en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur

Speculative:
  Time: 26832.2ms
  Output: for timothy was a spoiled cat and he allowed no one to interfere everybody waited upon him moving their chairs even for he was monarch of the hearth dinarzade la nuit suivante appela sa soeur quand il en fut temps si vous ne dormez pas ma soeur lui dit-elle je vous prie en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur

Speedup: 0.97x


## Tuning Thresholds

The two key thresholds control the trade-off between speed and accuracy:

- **CTC_THRESHOLD**: Lower = fewer CTC acceptances, more LLM verification
- **CONFIDENCE_THRESHOLD**: Lower = more LLM acceptances, fewer AR fallbacks

Experiment with these on your data to find the optimal balance.

In [34]:
# Experiment with different thresholds
threshold_configs = [
    (3.0, 0.1),   # Aggressive: accept more from CTC
    (1.0, 0.1),   # Balanced
    (0.7, 0.2),   # Conservative: verify more with LLM
]

print("Threshold Experiment")
print("=" * 60)

for ctc_thresh, conf_thresh in threshold_configs:
    # Temporarily override thresholds
    global CTC_THRESHOLD, CONFIDENCE_THRESHOLD
    CTC_THRESHOLD = ctc_thresh
    CONFIDENCE_THRESHOLD = conf_thresh
    
    predictions, stats = speculative_transcribe([audio])
    
    print(f"\nCTC={ctc_thresh}, Conf={conf_thresh}:")
    print(f"  Time: {stats['total_time']*1000:.1f}ms")
    print(f"  Path: CTC={stats['ctc_accepted']}, LLM={stats['llm_accepted']}, AR={stats['fallback_count']}")

# Reset to defaults
CTC_THRESHOLD = 0.5
CONFIDENCE_THRESHOLD = 0.01

Threshold Experiment

CTC=3.0, Conf=0.1:
  Time: 2382.8ms
  Path: CTC=1, LLM=0, AR=0

CTC=1.0, Conf=0.1:
  Time: 26816.6ms
  Path: CTC=0, LLM=0, AR=1

CTC=0.7, Conf=0.2:
  Time: 27053.4ms
  Path: CTC=0, LLM=0, AR=1


## Summary

Self-speculative decoding for speech recognition:

1. **CTC provides fast drafts** with confidence (entropy)
2. **LLM verifies** in a single forward pass
3. **AR fallback** only when needed

Benefits:
- Significant speedup for "easy" utterances
- Same quality as full AR for "hard" utterances
- Tunable speed/quality trade-off via thresholds